# Experiment 1b: Frozen SapBERT — Enriched Concept Index (No Translation)

**Goal:** Same as Experiment 1, but the FAISS concept index is enriched with **all training-set surface forms** per URI in addition to the canonical ICD-11 label.  
**Condition:** Traditional Chinese surface forms used **as-is** — no OpenCC conversion, no translation (identical to Experiment 1).  
**Corpora:** `english_ncbi`, `simp_chinese`, `trad_chinese`  
**Key difference from Experiment 1:** Each unique URI is represented by multiple vectors in FAISS (one per distinct surface form + canonical label). Top-10 retrieval deduplicates by URI before scoring.  
**Model:** Completely frozen throughout — `model.eval()` + `torch.no_grad()` everywhere, no optimizer, no gradient updates.

## GPU Check

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Please enable a GPU runtime in Colab: "
        "Runtime -> Change runtime type -> Hardware accelerator -> GPU."
    )

print(f"GPU available: {torch.cuda.get_device_name(0)}")
print(f"CUDA version:  {torch.version.cuda}")

GPU available: Tesla T4
CUDA version:  12.8


## Step 1 — Setup

Install dependencies and configure paths.

In [2]:
!pip install -q transformers faiss-cpu

In [18]:
# Mount Google Drive so we can read the corpus file.
# After running this cell, authorise access in the popup.
from google.colab import drive
drive.mount('/content/drive')

# Verify the mount succeeded
import os
print(os.listdir('/content/drive/MyDrive'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Getting started.pdf', '5.\xa0Acceptance\xa0of\xa0Project\xa0-\xa0_80\xa0(_3.5M).pdf', '5.\xa0Acceptance\xa0of\xa0Project\xa0-\xa0_80\xa0(_3.5M).gdoc', 'UCSD SOP.docx', 'Q.Q from drive', 'Berkeley', 'GENKI 1.zip', 'opt-20220603T170706Z-001.zip', 'R notebook-20220603T171726Z-001.zip', 'qpcr templates-20220603T171726Z-001.zip', 'Protocols-20220603T171721Z-001.zip', 'rsem_script(1)', 'rsem_script', 'guides', 'Weichen_Rsem_scripts', 'Windsor Village at Waltham - 5203-quote.pdf', 'PowerDVD18.0.1815.62（极致蓝光版）.zip', 'DVDFab12.01.7.X64.zip', 'Weichen Zhao resume.gdoc', 'COSI103A SWE Project', 'Workout plan.gdoc', 'ALC 080 LP_K1 Package_Weichen Zhao_2024_02-24-25.pdf', 'Copy of Agile Project Infographics by Slidesgo.gslides', 'Copy of Elegant Lines Pitch Deck | by Slidesgo.gslides', '2025_BIONER_project', 'Zhao_Weichen_resume_2025_fAIshion.pdf', 'Colab Notebooks', 'H

In [4]:
import sys
import transformers
import faiss
import numpy as np
import pandas as pd

print(f"Python:       {sys.version.split()[0]}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"faiss:        {faiss.__version__}")
print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")

Python:       3.12.13
torch:        2.10.0+cu128
transformers: 5.0.0
faiss:        1.13.2
numpy:        2.0.2
pandas:       2.2.2


In [30]:
import os

# Set REPO_ROOT to the folder in your Google Drive that contains
# processed_corpus/ and experiments/
REPO_ROOT = "/content/drive/MyDrive/2026_Medical_Entity_Linking_project/"  # <-- adjust if needed

DATA_PATH = os.path.join(
    REPO_ROOT,
    "data/no_translation/combined_disease_corpus_train_with_cuis_icd11_cleaned.jsonl"
)

OUTPUT_DIR    = os.path.join(REPO_ROOT, "experiments/frozen_sapbert/v2")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_JSONL = os.path.join(OUTPUT_DIR, "baseline_results_exp1_enriched_index.jsonl")
SUMMARY_CSV   = os.path.join(OUTPUT_DIR, "baseline_summary_exp1_enriched_index.csv")

SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BATCH_SIZE    = 32
DEVICE        = torch.device("cuda")

assert os.path.exists(DATA_PATH), (
    f"Data file not found: {DATA_PATH}\n"
    "Check that REPO_ROOT is set correctly and the file exists in Google Drive."
)

print(f"Data path:  {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device:     {DEVICE}")


Data path:  /content/drive/MyDrive/2026_Medical_Entity_Linking_project/data/no_translation/combined_disease_corpus_train_with_cuis_icd11_cleaned.jsonl
Output dir: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v2
Device:     cuda


## Step 2 — Load and Filter Data

In [7]:
import json
from collections import defaultdict

evaluable = []   # entities with a non-null ontology_id
skipped   = []   # entities with ontology_id == null

with open(DATA_PATH, "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        corpus = record["source_corpus"]
        for ent in record.get("entities", []):
            row = {
                "entity_id":      ent["entity_id"],
                "surface_form":   ent["surface_form"],
                "source_corpus":  corpus,
                "ontology_id":    ent.get("ontology_id"),
                "ontology_label": ent.get("ontology_label"),
            }
            if ent.get("ontology_id") is None:
                skipped.append(row)
            else:
                evaluable.append(row)

def corpus_counts(records):
    counts = defaultdict(int)
    for r in records:
        counts[r["source_corpus"]] += 1
    return dict(counts)

print(f"Evaluable entities (non-null ontology_id): {len(evaluable):,}")
print(f"  by corpus: {corpus_counts(evaluable)}")
print()
print(f"Skipped entities (null ontology_id): {len(skipped):,}")
print(f"  by corpus: {corpus_counts(skipped)}")

Evaluable entities (non-null ontology_id): 18,264
  by corpus: {'english_ncbi': 2158, 'simp_chinese': 13425, 'trad_chinese': 2681}

Skipped entities (null ontology_id): 12,136
  by corpus: {'english_ncbi': 797, 'simp_chinese': 5946, 'trad_chinese': 5393}


### Index Construction: Experiment 1 vs Experiment 1b

| | Experiment 1 | Experiment 1b (this notebook) |
|---|---|---|
| Vectors per URI | 1 (canonical label only) | 1 + all training surface forms |
| FAISS index size | = number of unique URIs | ≥ number of unique URIs |
| Retrieval | Top-10 vectors = top-10 URIs | Top-10 unique URIs after dedup |
| Mention coverage | Canonical labels only | Canonical + seen surface forms |

**Motivation:** The original SapBERT paper (Liu et al., 2021) trains on UMLS, whose concept index is rich with synonyms — each CUI has multiple preferred terms, aliases, and cross-lingual variants. A single-label index underrepresents that density. By adding all training-set surface forms as additional index vectors we give the retriever more anchor points per concept, which should particularly help for Chinese mentions whose script differs from the canonical English ICD-11 label.

> Note: this enrichment uses only the training split of our corpus, so there is no leakage — the gold labels used for evaluation are the same `ontology_id` values; we are not enriching with test surface forms.

## Step 3 — Build Enriched Concept Index

For each unique URI collect:
- The canonical `ontology_label` from the data
- All distinct `surface_form` values that appear in the training set with that URI

Each text representation is encoded separately with frozen SapBERT (mean pooling → L2 normalise).  
All vectors for a URI are added to the FAISS `IndexFlatIP` index, so the index contains **more vectors than unique URIs**.  
A mapping `faiss_int → ontology_id` enables URI retrieval for any vector.

In [8]:
from transformers import AutoTokenizer, AutoModel

print(f"Loading SapBERT from '{SAPBERT_MODEL}' ...")
tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
model     = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE)

model.eval()
for param in model.parameters():
    param.requires_grad = False

print("Model loaded and completely frozen.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading SapBERT from 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext' ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded and completely frozen.
Total parameters: 109,482,240


In [9]:
def encode_texts(texts, batch_size=BATCH_SIZE):
    """Return L2-normalised float32 embeddings for a list of strings."""
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=64,
                return_tensors="pt",
            ).to(DEVICE)
            output = model(**encoded)
            # Mean pooling over token positions
            attention_mask   = encoded["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)
            token_embeddings = output.last_hidden_state                          # (B, T, H)
            summed    = (token_embeddings * attention_mask).sum(dim=1)           # (B, H)
            counts    = attention_mask.sum(dim=1)                                # (B, 1)
            mean_pooled = summed / counts                                        # (B, H)
            all_embeddings.append(mean_pooled.cpu().float().numpy())
    embeddings = np.concatenate(all_embeddings, axis=0)  # (N, H)
    # L2 normalise
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return (embeddings / norms).astype(np.float32)

In [10]:
# Build per-URI text representation sets
# Key: ontology_id  →  ordered list of unique text strings
# Canonical label always comes first; training surface forms are appended.
uri_to_canonical = {}   # ontology_id -> canonical ICD-11 label
uri_to_surfaces  = {}   # ontology_id -> set of training surface forms

for ent in evaluable:
    oid   = ent["ontology_id"]
    label = ent["ontology_label"] or ""
    sf    = ent["surface_form"]
    if oid not in uri_to_canonical:
        uri_to_canonical[oid] = label
        uri_to_surfaces[oid]  = set()
    uri_to_surfaces[oid].add(sf)

# Build flat parallel lists for encoding
# index_to_id[i] = ontology_id for FAISS vector i
all_texts   = []
index_to_id = []

for oid in uri_to_canonical:
    canonical = uri_to_canonical[oid]
    surfaces  = uri_to_surfaces[oid]
    # Deduplicate: canonical label first, then any surface form not identical to it
    texts_for_uri = [canonical] + [s for s in sorted(surfaces) if s != canonical]
    for t in texts_for_uri:
        all_texts.append(t)
        index_to_id.append(oid)

# ── Summary stats ────────────────────────────────────────────────────────
uri_vec_counts = {}
for oid in index_to_id:
    uri_vec_counts[oid] = uri_vec_counts.get(oid, 0) + 1

counts = list(uri_vec_counts.values())
import numpy as _np
bins = {"1": 0, "2-5": 0, "6-10": 0, "10+": 0}
for c in counts:
    if   c == 1:    bins["1"]    += 1
    elif c <= 5:    bins["2-5"]  += 1
    elif c <= 10:   bins["6-10"] += 1
    else:           bins["10+"]  += 1

print(f"Unique URIs:             {len(uri_vec_counts):,}")
print(f"Total vectors in index:  {len(all_texts):,}")
print(f"Avg vectors per URI:     {_np.mean(counts):.2f}")
print(f"URIs with 1 surface form:    {bins['1']:,}")
print(f"URIs with 2-5 surface forms: {bins['2-5']:,}")
print(f"URIs with 6-10 surface forms:{bins['6-10']:,}")
print(f"URIs with 10+ surface forms: {bins['10+']:,}")

# ── Encode and build FAISS index ─────────────────────────────────────────
print("\nEncoding all concept representations with frozen SapBERT ...")
concept_embeddings = encode_texts(all_texts)
print(f"Concept embeddings shape: {concept_embeddings.shape}")

dim   = concept_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(concept_embeddings)
print(f"FAISS IndexFlatIP built with {index.ntotal:,} vectors (dim={dim}).")


Unique URIs:             2,931
Total vectors in index:  6,263
Avg vectors per URI:     2.14
URIs with 1 surface form:    562
URIs with 2-5 surface forms: 2,340
URIs with 6-10 surface forms:27
URIs with 10+ surface forms: 2

Encoding all concept representations with frozen SapBERT ...
Concept embeddings shape: (6263, 768)
FAISS IndexFlatIP built with 6,263 vectors (dim=768).


## Step 4 — Encode Mentions

Encode each evaluable entity's `surface_form` with the same frozen SapBERT encoder.  
**Experiment 1 condition:** Traditional Chinese surface forms are used **as-is** (no OpenCC, no translation).

In [11]:
print(f"Encoding {len(evaluable):,} mention surface forms with frozen SapBERT ...")

surface_forms      = [ent["surface_form"] for ent in evaluable]
mention_embeddings = encode_texts(surface_forms)

print(f"Mention embeddings shape: {mention_embeddings.shape}")

Encoding 18,264 mention surface forms with frozen SapBERT ...
Mention embeddings shape: (18264, 768)


## Step 5 — Retrieve and Evaluate

Retrieve top-10 candidates per mention from the FAISS index.  
Compute Acc@1, Acc@5, Acc@10 for the full dataset and per source corpus.

In [12]:
K        = 10    # desired number of unique URIs per mention
K_FETCH  = K * 10  # over-fetch to ensure K unique URIs after dedup

print(f"Searching FAISS index (fetch {K_FETCH} vectors, dedup to top-{K} unique URIs) "
      f"for {len(evaluable):,} mentions ...")
_scores, _indices = index.search(mention_embeddings, K_FETCH)  # (N, K_FETCH)
print("Search complete.")


Searching FAISS index (fetch 100 vectors, dedup to top-10 unique URIs) for 18,264 mentions ...
Search complete.


In [13]:
per_entity_results = []

for i, ent in enumerate(evaluable):
    gold_uri = ent["ontology_id"]

    # Deduplicate by URI while preserving rank order
    seen     = set()
    retrieved = []
    for idx in _indices[i]:
        uri = index_to_id[idx]
        if uri not in seen:
            seen.add(uri)
            retrieved.append(uri)
        if len(retrieved) == K:
            break

    top1_correct  = retrieved[0] == gold_uri
    top5_correct  = gold_uri in retrieved[:5]
    top10_correct = gold_uri in retrieved[:10]

    per_entity_results.append({
        "entity_id":          ent["entity_id"],
        "surface_form":       ent["surface_form"],
        "source_corpus":      ent["source_corpus"],
        "gold_uri":           gold_uri,
        "top1_predicted_uri": retrieved[0],
        "top1_correct":       top1_correct,
        "top5_correct":       top5_correct,
        "top10_correct":      top10_correct,
        "top5_candidates":    retrieved[:5],
    })

print(f"Evaluation complete for {len(per_entity_results):,} entities.")


Evaluation complete for 18,264 entities.


In [14]:
def compute_acc(records):
    n = len(records)
    if n == 0:
        return {"N": 0, "Acc@1": float("nan"), "Acc@5": float("nan"), "Acc@10": float("nan")}
    return {
        "N":      n,
        "Acc@1":  round(sum(r["top1_correct"]  for r in records) / n, 4),
        "Acc@5":  round(sum(r["top5_correct"]  for r in records) / n, 4),
        "Acc@10": round(sum(r["top10_correct"] for r in records) / n, 4),
    }

rows = {}
for corpus in ["english_ncbi", "simp_chinese", "trad_chinese"]:
    subset = [r for r in per_entity_results if r["source_corpus"] == corpus]
    rows[corpus] = compute_acc(subset)
rows["ALL"] = compute_acc(per_entity_results)

summary_df = pd.DataFrame(rows).T[["N", "Acc@1", "Acc@5", "Acc@10"]]
summary_df.index.name = "corpus"

print("\n=== Experiment 1: Frozen SapBERT — No Translation ===")
print(summary_df.to_string())


=== Experiment 1: Frozen SapBERT — No Translation ===
                    N   Acc@1   Acc@5  Acc@10
corpus                                       
english_ncbi   2158.0  0.5760  0.6687  0.7442
simp_chinese  13425.0  0.6335  0.7835  0.8094
trad_chinese   2681.0  0.6822  0.7773  0.8146
ALL           18264.0  0.6339  0.7691  0.8025


In [15]:
summary_df

,N,Acc@1,Acc@5,Acc@10
corpus,,,,
english_ncbi,2158.0,0.5760,0.6687,0.7442
simp_chinese,13425.0,0.6335,0.7835,0.8094
trad_chinese,2681.0,0.6822,0.7773,0.8146
ALL,18264.0,0.6339,0.7691,0.8025


## Step 6 — Save Results

In [16]:
# Per-entity JSONL
with open(RESULTS_JSONL, "w", encoding="utf-8") as fh:
    for row in per_entity_results:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Per-entity results saved to: {RESULTS_JSONL}")

# Summary CSV
summary_df.to_csv(SUMMARY_CSV)
print(f"Summary table saved to:      {SUMMARY_CSV}")

Per-entity results saved to: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/v2/frozen_sapbert/baseline_results_exp1_enriched_index.jsonl
Summary table saved to:      /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/v2/frozen_sapbert/baseline_summary_exp1_enriched_index.csv


---
## Step 7 — Evaluate on Test Set

Load the held-out test JSONL, filter to evaluable entities (non-null `ontology_id`), encode mentions with the same frozen SapBERT encoder, and retrieve from the **already-built enriched FAISS index** (no re-encoding of concepts needed).  
Report Acc@1, Acc@5, Acc@10 for the full test set and per source corpus.  
Save per-entity results and a summary CSV.

> The concept index is unchanged — it was built from the training split only.

In [33]:
TEST_DATA_PATH = os.path.join(
    REPO_ROOT,
    "data/no_translation/combined_disease_corpus_test_with_cuis_icd11_cleaned.jsonl"
)

TEST_RESULTS_JSONL = os.path.join(OUTPUT_DIR, "test_results_exp1b_enriched_index.jsonl")
TEST_SUMMARY_CSV   = os.path.join(OUTPUT_DIR, "test_summary_exp1b_enriched_index.csv")

assert os.path.exists(TEST_DATA_PATH), (
    f"Test file not found: {TEST_DATA_PATH}\n"
    "Check that REPO_ROOT is set correctly and the file exists in Google Drive."
)
print(f"Test data path: {TEST_DATA_PATH}")
print(f"Test data path: {TEST_RESULTS_JSONL}")


Test data path: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/data/no_translation/combined_disease_corpus_test_with_cuis_icd11_cleaned.jsonl
Test data path: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v2/test_results_exp1b_enriched_index.jsonl


In [20]:
test_evaluable = []
test_skipped   = []

with open(TEST_DATA_PATH, "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        corpus = record["source_corpus"]
        for ent in record.get("entities", []):
            row = {
                "entity_id":      ent["entity_id"],
                "surface_form":   ent["surface_form"],
                "source_corpus":  corpus,
                "ontology_id":    ent.get("ontology_id"),
                "ontology_label": ent.get("ontology_label"),
            }
            if ent.get("ontology_id") is None:
                test_skipped.append(row)
            else:
                test_evaluable.append(row)

print(f"Test evaluable entities: {len(test_evaluable):,}")
print(f"  by corpus: {corpus_counts(test_evaluable)}")
print()
print(f"Test skipped (null ontology_id): {len(test_skipped):,}")
print(f"  by corpus: {corpus_counts(test_skipped)}")


Test evaluable entities: 3,311
  by corpus: {'english_ncbi': 361, 'simp_chinese': 2560, 'trad_chinese': 390}

Test skipped (null ontology_id): 1,710
  by corpus: {'english_ncbi': 195, 'simp_chinese': 900, 'trad_chinese': 615}


In [21]:
print(f"Encoding {len(test_evaluable):,} test mention surface forms with frozen SapBERT ...")

test_surface_forms      = [ent["surface_form"] for ent in test_evaluable]
test_mention_embeddings = encode_texts(test_surface_forms)

print(f"Test mention embeddings shape: {test_mention_embeddings.shape}")


Encoding 3,311 test mention surface forms with frozen SapBERT ...
Test mention embeddings shape: (3311, 768)


In [22]:
print(f"Searching enriched FAISS index (fetch {K_FETCH} vectors, dedup to top-{K} unique URIs) "
      f"for {len(test_evaluable):,} test mentions ...")
_test_scores, _test_indices = index.search(test_mention_embeddings, K_FETCH)  # (N, K_FETCH)
print("Search complete.")


Searching enriched FAISS index (fetch 100 vectors, dedup to top-10 unique URIs) for 3,311 test mentions ...
Search complete.


In [23]:
test_per_entity_results = []

for i, ent in enumerate(test_evaluable):
    gold_uri = ent["ontology_id"]

    # Deduplicate by URI while preserving rank order
    seen      = set()
    retrieved = []
    for idx in _test_indices[i]:
        uri = index_to_id[idx]
        if uri not in seen:
            seen.add(uri)
            retrieved.append(uri)
        if len(retrieved) == K:
            break

    top1_correct  = retrieved[0] == gold_uri
    top5_correct  = gold_uri in retrieved[:5]
    top10_correct = gold_uri in retrieved[:10]

    test_per_entity_results.append({
        "entity_id":          ent["entity_id"],
        "surface_form":       ent["surface_form"],
        "source_corpus":      ent["source_corpus"],
        "gold_uri":           gold_uri,
        "top1_predicted_uri": retrieved[0],
        "top1_correct":       top1_correct,
        "top5_correct":       top5_correct,
        "top10_correct":      top10_correct,
        "top5_candidates":    retrieved[:5],
    })

print(f"Test evaluation complete for {len(test_per_entity_results):,} entities.")


Test evaluation complete for 3,311 entities.


In [24]:
test_rows = {}
for corpus in ["english_ncbi", "simp_chinese", "trad_chinese"]:
    subset = [r for r in test_per_entity_results if r["source_corpus"] == corpus]
    test_rows[corpus] = compute_acc(subset)
test_rows["ALL"] = compute_acc(test_per_entity_results)

test_summary_df = pd.DataFrame(test_rows).T[["N", "Acc@1", "Acc@5", "Acc@10"]]
test_summary_df.index.name = "corpus"

print("\n=== Experiment 1b TEST SET: Frozen SapBERT — Enriched Index, No Translation ===")
print(test_summary_df.to_string())



=== Experiment 1b TEST SET: Frozen SapBERT — Enriched Index, No Translation ===
                   N   Acc@1   Acc@5  Acc@10
corpus                                      
english_ncbi   361.0  0.3961  0.4571  0.5014
simp_chinese  2560.0  0.4793  0.6230  0.6426
trad_chinese   390.0  0.6256  0.7385  0.7974
ALL           3311.0  0.4875  0.6185  0.6454


In [25]:
test_summary_df


,N,Acc@1,Acc@5,Acc@10
corpus,,,,
english_ncbi,361.0,0.3961,0.4571,0.5014
simp_chinese,2560.0,0.4793,0.6230,0.6426
trad_chinese,390.0,0.6256,0.7385,0.7974
ALL,3311.0,0.4875,0.6185,0.6454


### Train vs Test Accuracy Comparison

In [26]:
comparison = pd.concat(
    [summary_df.add_suffix(" (train)"), test_summary_df.add_suffix(" (test)")],
    axis=1,
)[["N (train)", "Acc@1 (train)", "N (test)", "Acc@1 (test)",
   "Acc@5 (train)", "Acc@5 (test)", "Acc@10 (train)", "Acc@10 (test)"]]
comparison.index.name = "corpus"
print(comparison.to_string())
comparison


              N (train)  Acc@1 (train)  N (test)  Acc@1 (test)  Acc@5 (train)  Acc@5 (test)  Acc@10 (train)  Acc@10 (test)
corpus                                                                                                                    
english_ncbi     2158.0         0.5760     361.0        0.3961         0.6687        0.4571          0.7442         0.5014
simp_chinese    13425.0         0.6335    2560.0        0.4793         0.7835        0.6230          0.8094         0.6426
trad_chinese     2681.0         0.6822     390.0        0.6256         0.7773        0.7385          0.8146         0.7974
ALL             18264.0         0.6339    3311.0        0.4875         0.7691        0.6185          0.8025         0.6454


,N (train),Acc@1 (train),N (test),Acc@1 (test),Acc@5 (train),Acc@5 (test),Acc@10 (train),Acc@10 (test)
corpus,,,,,,,,
english_ncbi,2158.0,0.5760,361.0,0.3961,0.6687,0.4571,0.7442,0.5014
simp_chinese,13425.0,0.6335,2560.0,0.4793,0.7835,0.6230,0.8094,0.6426
trad_chinese,2681.0,0.6822,390.0,0.6256,0.7773,0.7385,0.8146,0.7974
ALL,18264.0,0.6339,3311.0,0.4875,0.7691,0.6185,0.8025,0.6454


In [34]:
# Per-entity JSONL
with open(TEST_RESULTS_JSONL, "w", encoding="utf-8") as fh:
    for row in test_per_entity_results:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Test per-entity results saved to: {TEST_RESULTS_JSONL}")

# Summary CSV
test_summary_df.to_csv(TEST_SUMMARY_CSV)
print(f"Test summary table saved to:      {TEST_SUMMARY_CSV}")


Test per-entity results saved to: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v2/test_results_exp1b_enriched_index.jsonl
Test summary table saved to:      /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert/v2/test_summary_exp1b_enriched_index.csv


---
## Next: Experiment 2

**Experiment 2** will repeat the Experiment 1 setup (single canonical label per URI) with one change: Traditional Chinese surface forms will be converted to Simplified Chinese using **OpenCC** (`t2s` conversion) before encoding with SapBERT. This tests whether script normalisation improves linking accuracy for the `trad_chinese` corpus.

A natural follow-up would be **Experiment 2b**: combine the enriched index from Experiment 1b with the OpenCC-converted mention encoding from Experiment 2.